# Autoencoder denso, NSL-KDD

Autoencoder denso simétrico con un cuello de botella de 16 neuronas, entrenado solo con tráfico normal. Una muestra se clasifica como ataque si su error de reconstrucción supera el percentil 95 del error de entrenamiento. Se entrena con KDDTrain+_20Percent y se evalúa con KDDTest+, que incluye tipos de ataque no vistos en entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
BASE = Path.home() / ".cache/kagglehub/datasets/hassan06/nslkdd/versions/1"
df_train = pd.read_csv(BASE / "KDDTrain+_20Percent.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])
df_test = pd.read_csv(BASE / "KDDTest+.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])

df_train['is_attack'] = (df_train['label'] != 'normal').astype(int)
df_test['is_attack'] = (df_test['label'] != 'normal').astype(int)
df_train['_split'] = 'train'
df_test['_split'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

# One-hot conjunto de train y test, porque hay valores de service exclusivos de cada fichero
X_all = pd.get_dummies(df_all.drop(columns=['label', 'is_attack', '_split']), columns=['protocol_type', 'service', 'flag'])
is_train = (df_all['_split'] == 'train').to_numpy()

X_train_df = X_all[is_train].reset_index(drop=True)
X_test_df = X_all[~is_train].reset_index(drop=True)
y_train = df_train['is_attack'].to_numpy()
y_test = df_test['is_attack'].to_numpy()
tipos_nuevos = set(df_test.loc[df_test['is_attack'] == 1, 'label'].unique()) - set(df_train.loc[df_train['is_attack'] == 1, 'label'].unique())
print(f"X_train: {X_train_df.shape}, X_test: {X_test_df.shape}")
print(f"Ataques train: {y_train.sum()}/{len(y_train)}, ataques test: {y_test.sum()}/{len(y_test)}")
print(f"Tipos de ataque en test nunca vistos en train: {len(tipos_nuevos)} -> {sorted(tipos_nuevos)}")

X_train: (25192, 119), X_test: (22544, 119)
Ataques train: 11743/25192, ataques test: 12833/22544
Tipos de ataque en test nunca vistos en train: 18 -> ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'perl', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


In [3]:
scaler = MinMaxScaler().fit(X_train_df.to_numpy())
X_train = scaler.transform(X_train_df.to_numpy())
X_test = scaler.transform(X_test_df.to_numpy())
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (25192, 119), X_train_normal: (13449, 119), X_test: (22544, 119)


In [4]:
input_dim = X_train.shape[1]

inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(input_dim, activation='sigmoid')(x)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f"Epocas entrenadas: {len(history.history['loss'])}, val_loss final: {history.history['val_loss'][-1]:.5f}")

Epocas entrenadas: 100, val_loss final: 0.00165


In [5]:
def reconstruction_error(X):
    return np.mean(np.square(X - autoencoder.predict(X, verbose=0)), axis=1)

threshold = np.percentile(reconstruction_error(X_train_normal), 95)
y_pred = (reconstruction_error(X_test) > threshold).astype(int)

print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, reconstruction_error(X_test)):.4f}")

# Error de test para el histograma de figuras/figuras_cap5.py
np.savez("err_nsl.npz", e_te=reconstruction_error(X_test), y=y_test, thr=threshold)

              precision    recall  f1-score   support

      normal       0.75      0.93      0.83      9711
      ataque       0.94      0.77      0.84     12833

    accuracy                           0.84     22544
   macro avg       0.84      0.85      0.84     22544
weighted avg       0.86      0.84      0.84     22544

[[9037  674]
 [2979 9854]]


ROC-AUC: 0.9406
